# 11 — Measure key events


This notebook also exports the exact waveform, baseline, and signal windows used for each named-event measurement so downstream figure notebooks do not reconstruct them.


Measure positive, negative, peak-to-peak, and reduced pressure for the key
arrivals using the canonical Notebook 02 moving-median-baseline-corrected
pressure Stream. The short local baseline windows used by the measurement
helper are retained only to remove any small residual constant offset.


This compact notebook provides named-event measurements used in the manuscript
and figure annotations. It remains separate from Notebook 10 because these
windows are manually defined for a few physically important arrivals rather
than derived from the full event catalogue.


In [1]:
from pathlib import Path
import json
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks2":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime

from modules import project_config as config

DERIVED_DIR = config.DERIVED_DIR
FIGURE_DIR = config.FIGURE_DIR

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [2]:
from obspy import read
from modules.event_measurements import (
    measure_reduced_pressures_in_window,
    pressure_results_for_paper,
)

ANALYSIS_CONFIG_FILE = (
    config.OUTPUT_DIR / "02_analysis_configuration.json"
)

if not ANALYSIS_CONFIG_FILE.exists():
    raise FileNotFoundError(ANALYSIS_CONFIG_FILE)

analysis_config = json.loads(
    ANALYSIS_CONFIG_FILE.read_text()
)

GEOMETRY_FILE = Path(
    analysis_config["geometry_file"]
).expanduser()

if not GEOMETRY_FILE.exists():
    raise FileNotFoundError(
        "Geometry product recorded by Notebook 02 was not found: "
        f"{GEOMETRY_FILE}"
    )

baseline_products = analysis_config.get(
    "baseline_removed_streams",
    {},
)
BASELINE_STREAM_FILE = Path(
    baseline_products.get(
        "pickle",
        DERIVED_DIR
        / "bchh_corrected_moving_median_baseline_removed.pkl",
    )
).expanduser()

if not BASELINE_STREAM_FILE.exists():
    raise FileNotFoundError(
        "Run Notebook 02 to create the baseline-corrected Stream: "
        f"{BASELINE_STREAM_FILE}"
    )

st_corr = read(
    str(BASELINE_STREAM_FILE),
    format="PICKLE",
)

geometry = pd.read_csv(GEOMETRY_FILE)
geometry["channel"] = (
    geometry["channel"]
    .astype(str)
    .str.upper()
)

pressure_channels = tuple(
    analysis_config["pressure_channels"]
)

noncanonical_channels = [
    channel
    for channel in pressure_channels
    if not channel.startswith("DD")
]
if noncanonical_channels:
    raise ValueError(
        "Notebook 02 configuration contains noncanonical pressure "
        f"channels: {noncanonical_channels}"
    )

SENSOR_DISTANCES_M = (
    geometry.loc[
        geometry["channel"].isin(pressure_channels)
    ]
    .set_index("channel")["distance_m"]
    .astype(float)
    .to_dict()
)

missing_geometry_channels = (
    set(pressure_channels) - set(SENSOR_DISTANCES_M)
)
if missing_geometry_channels:
    raise KeyError(
        "Geometry table lacks canonical pressure channels: "
        f"{sorted(missing_geometry_channels)}"
    )

stream_channels = {
    trace.stats.channel
    for trace in st_corr
}
missing_stream_channels = (
    set(pressure_channels) - stream_channels
)
if missing_stream_channels:
    raise KeyError(
        "Notebook 02 baseline Stream lacks canonical pressure "
        f"channels: {sorted(missing_stream_channels)}"
    )

print("Pressure source:", BASELINE_STREAM_FILE)
print("Geometry:", GEOMETRY_FILE)
print("Pressure channels:", pressure_channels)
print("Sensor distances (m):", SENSOR_DISTANCES_M)


Pressure source: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/derived2/bchh_corrected_moving_median_baseline_removed.pkl
Geometry: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/outputs2/02_bchh_channels.csv
Pressure channels: ('DD1', 'DD2', 'DD3')
Sensor distances (m): {'DD1': 1439.9638305217552, 'DD2': 1408.3384566901343, 'DD3': 1412.8543304792386}


## Measurement windows

In [3]:

EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.080")

event_windows = [
    {
        "event": "Initial second-stage failure",
        "start": EXPLOSION_TIME + 3.0 - 0.08,
        "end": EXPLOSION_TIME + 5.0 - 0.08,
        "baseline_start_s": 0.10,
        "baseline_end_s": 0.80,
        "signal_start_s": 0.85,
        "signal_end_s": 1.40,
    },
    {
        "event": "Principal explosion",
        "start": EXPLOSION_TIME + 6.0 - 0.08,
        "end": EXPLOSION_TIME + 9.0 - 0.08,
        "baseline_start_s": 0.10,
        "baseline_end_s": 0.75,
        "signal_start_s": 0.75,
        "signal_end_s": 2.50,
    },
    {
        "event": "Capsule pulse 1",
        "start": UTCDateTime("2016-09-01T13:07:28.30"),
        "end": UTCDateTime("2016-09-01T13:07:28.75"),
        "baseline_start_s": 0.00,
        "baseline_end_s": 0.12,
        "signal_start_s": 0.12,
        "signal_end_s": 0.45,
    },
    {
        "event": "Capsule pulse 2",
        "start": UTCDateTime("2016-09-01T13:07:28.85"),
        "end": UTCDateTime("2016-09-01T13:07:29.25"),
        "baseline_start_s": 0.00,
        "baseline_end_s": 0.10,
        "signal_start_s": 0.10,
        "signal_end_s": 0.40,
    },
]


In [4]:
all_results = []
event_streams = []

window_rows = []

for spec in event_windows:
    event_stream, result = measure_reduced_pressures_in_window(
        st_corr,
        spec["start"],
        spec["end"],
        SENSOR_DISTANCES_M,
        reference_distance_m=1000.0,
        event_name=spec["event"],
        baseline_start_s=spec["baseline_start_s"],
        baseline_end_s=spec["baseline_end_s"],
        signal_start_s=spec["signal_start_s"],
        signal_end_s=spec["signal_end_s"],
    )

    event_streams.append(event_stream)
    all_results.append(result)

    window_rows.append({
        "event": spec["event"],
        "window_start_utc": spec["start"].isoformat(),
        "window_end_utc": spec["end"].isoformat(),
        "window_start_epoch_s": float(spec["start"].timestamp),
        "window_end_epoch_s": float(spec["end"].timestamp),
        "window_duration_s": float(spec["end"] - spec["start"]),
        "baseline_start_s": float(spec["baseline_start_s"]),
        "baseline_end_s": float(spec["baseline_end_s"]),
        "signal_start_s": float(spec["signal_start_s"]),
        "signal_end_s": float(spec["signal_end_s"]),
        "reference_distance_m": 1000.0,
        "waveform_product": str(BASELINE_STREAM_FILE),
        "geometry_product": str(GEOMETRY_FILE),
    })

key_event_pressures = pd.concat(
    all_results,
    ignore_index=True,
)
key_event_windows = pd.DataFrame(window_rows)

pressure_output_file = (
    DERIVED_DIR / "key_event_pressure_measurements.csv"
)
window_output_file = (
    DERIVED_DIR / "key_event_measurement_windows.csv"
)
metadata_output_file = (
    DERIVED_DIR / "key_event_measurement_metadata.json"
)

key_event_pressures.to_csv(
    pressure_output_file,
    index=False,
)
key_event_windows.to_csv(
    window_output_file,
    index=False,
)

measurement_metadata = {
    "producer_notebook": "11_measure_key_events.ipynb",
    "analysis_configuration": str(ANALYSIS_CONFIG_FILE),
    "geometry_file": str(GEOMETRY_FILE),
    "waveform_product": str(BASELINE_STREAM_FILE),
    "pressure_channels": list(pressure_channels),
    "sensor_distances_m": {
        channel: float(distance)
        for channel, distance in SENSOR_DISTANCES_M.items()
    },
    "reference_distance_m": 1000.0,
    "pressure_measurements_file": str(pressure_output_file),
    "measurement_windows_file": str(window_output_file),
}
metadata_output_file.write_text(
    json.dumps(measurement_metadata, indent=2) + "\n"
)

print("Key-event pressure measurements:")
display(pressure_results_for_paper(key_event_pressures))

print("Authoritative key-event measurement windows:")
display(key_event_windows)

print("Wrote:", pressure_output_file)
print("Wrote:", window_output_file)
print("Wrote:", metadata_output_file)


Key-event pressure measurements:


,event,channel,distance_m,positive_peak_pa,negative_peak_pa,peak_to_peak_pa,positive_reduced_pa,negative_reduced_pa,peak_to_peak_reduced_pa
0,Initial second-stage failure,DD1,1440.0,30.4,-72.2,102.6,43.8,-104.0,147.8
1,Initial second-stage failure,DD2,1408.3,51.7,-25.4,77.1,72.8,-35.8,108.6
2,Initial second-stage failure,DD3,1412.9,38.8,-21.2,60.1,54.9,-30.0,84.9
3,Initial second-stage failure,MEDIAN,1412.9,38.8,-25.4,77.1,54.9,-35.8,108.6
4,Principal explosion,DD1,1440.0,1312.5,-127.4,1439.9,1890.0,-183.4,2073.4
5,Principal explosion,DD2,1408.3,1512.7,-158.2,1670.9,2130.4,-222.8,2353.1
6,Principal explosion,DD3,1412.9,1392.9,-156.5,1549.4,1968.0,-221.1,2189.0
7,Principal explosion,MEDIAN,1412.9,1392.9,-156.5,1549.4,1968.0,-221.1,2189.0
8,Capsule pulse 1,DD1,1440.0,238.7,-115.1,353.8,343.8,-165.8,509.5
9,Capsule pulse 1,DD2,1408.3,279.9,-114.6,394.5,394.1,-161.4,555.5


Authoritative key-event measurement windows:


,event,window_start_utc,window_end_utc,window_start_epoch_s,window_end_epoch_s,window_duration_s,baseline_start_s,baseline_end_s,signal_start_s,signal_end_s,reference_distance_m,waveform_product,geometry_product
0,Initial second-stage failure,2016-09-01T13:07:15,2016-09-01T13:07:17,1.472735e+09,1.472735e+09,2.00,0.1,0.80,0.85,1.40,1000.0,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/Library/CloudStorage/Box-Box/...
1,Principal explosion,2016-09-01T13:07:18,2016-09-01T13:07:21,1.472735e+09,1.472735e+09,3.00,0.1,0.75,0.75,2.50,1000.0,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/Library/CloudStorage/Box-Box/...
2,Capsule pulse 1,2016-09-01T13:07:28.300000,2016-09-01T13:07:28.750000,1.472735e+09,1.472735e+09,0.45,0.0,0.12,0.12,0.45,1000.0,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/Library/CloudStorage/Box-Box/...
3,Capsule pulse 2,2016-09-01T13:07:28.850000,2016-09-01T13:07:29.250000,1.472735e+09,1.472735e+09,0.40,0.0,0.10,0.10,0.40,1000.0,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/Library/CloudStorage/Box-Box/...


Wrote: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/derived2/key_event_pressure_measurements.csv
Wrote: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/derived2/key_event_measurement_windows.csv
Wrote: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/spacex_paper/data/derived2/key_event_measurement_metadata.json
